<a href="https://colab.research.google.com/github/CarlKo-DLSU/PowerpuffCarl-MP1/blob/main/MP_Problem4_PowerpuffCarl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MP1 - Problem 4 | Powerpuff Carl

This script numerically solves for parameters (θ, φ, λ) such that the Qiskit U3 gate:

    U(θ, φ, λ)

is equal to the Hadamard gate H up to global phase.

The program performs:
1. A coarse 3D grid search across (θ, φ, λ)
2. A local refinement around the best coarse grid point
3. A final Nelder–Mead optimization (local minimizer)
4. Validation by comparing NumPy and Qiskit unitary matrices, including:
   - global-phase-invariant unitary distance
   - action of the operator on |0⟩
   - consistency between analytic NumPy U3 and Qiskit U3 gate

The final result is a set of parameters accurate to ~10⁻¹⁵, verifying that:

    H = U3(π/2, 0, π)   //up to global phase.

# Install required quantum, math, and formatting libraries

This block installs the minimal set of Python packages needed:
qiskit: Used for two purposes:
1. Building a 1-qubit U(θ, Φ, λ) circuit to obtain the actual unitary
matrix that Qiskit uses internally.
2. Comparing the Qiskit-generated U matrix to our numpy formula.

- numpy:
    Needed for matrix algebra, trigonometric functions, and vector norms.

- pylatexenc:
    Used only to render LaTeX labels (U(θ, Φ, λ), etc.) more cleanly in text output.

- scipy:
    Used for local optimization (Nelder-Mead) to polish the grid search result.

In [ ]:
!pip install qiskit
!pip install numpy
!pip install pylatexenc
!pip install scipy

# Import core libraries
This block imports the tools used throughout the notebook:
- math, numpy:
    For scalar and matrix-level computations — these implement the analytic
    U(θ, Φ, λ) formula and enable the global-phase-invariant distance metric.

- pylatexenc:
    Converts LaTeX strings like "U(\theta,\Phi,\lambda)" into plain UTF-8 text so the
    printed output in the notebook looks clean.

- Qiskit QuantumCircuit + Operator:
    These are optional but extremely useful:
    
  - QuantumCircuit.u(θ, Φ, λ) constructs the real U3 gate used by Qiskit.

  - Operator(qc).data extracts the exact 2×2 matrix representation.
  
  - Using Qiskit's version ensures we are validating a real-world
  implementation of U3, not just our derived analytic formula.

- scipy.optimize.minimize:
    For polishing the grid search result with local optimization.

In [ ]:
import math
import numpy as np
from pylatexenc.latex2text import LatexNodes2Text
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from scipy.optimize import minimize

# Hadamard Gate Definition
Here we explicitly define the Hadamard gate:

    H = (1/√2) [[1, 1], [1, -1]]
We store it as a complex numpy matrix. This allows:
  - Matrix multiplication with U(θ, Φ, λ)
  - Comparison via global-phase-invariant distance
  - Evaluating its action on |0⟩, |1⟩
  
This is the "target" unitary our U(θ, Φ, λ) should match up to global phase.

In [ ]:
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

# Analytic U Matrix via Numpy
Implement the exact formula for U(θ, Φ, λ):

    U(θ, Φ, λ) = [[cos(θ/2), -e^{iλ} sin(θ/2)], [-e^{iΦ} sin(θ/2), -e^{i(Φ+λ)} cos(θ/2)]]
Here we construct the 2×2 matrix using numpy primitives. This serves two roles:
  1. Enables extremely fast evaluations during the grid + refine + polish search.
  2. Lets us compare a mathematically derived U gate with Qiskit's
     built-in U3 implementation for correctness checking.
     
Because U3 is a very common gate in quantum computing, validating analytic
vs Qiskit behavior is both instructive and useful in debugging.

In [ ]:
def u_matrix_numpy(theta: float, phi: float, lam: float) -> np.ndarray:
    """Qiskit-compatible U3 matrix definition."""
    c = math.cos(theta / 2.0)
    s = math.sin(theta / 2.0)
    return np.array([
        [c, -np.exp(1j * lam) * s],
        [np.exp(1j * phi) * s, np.exp(1j * (phi + lam)) * c]
    ], dtype=complex)

# Qiskit-Based U Matrix
This function generates the U(θ, Φ, λ) gate using Qiskit’s native implementation.

Steps:
- Initialize a 1-qubit QuantumCircuit.
- Apply qc.u(θ, Φ, λ) to qubit 0.
- Convert the entire circuit into a matrix using Operator(qc).data.

Purpose:
- Verifies that our analytic matrix matches Qiskit's hardware-oriented
implementation.
- Ensures absolute confidence that the U gate used in real devices obeys
the same mathematics as our numpy version.

This is a correctness and consistency check.

In [ ]:
def u_matrix_qiskit(theta: float, phi: float, lam: float) -> np.ndarray:
    """Build a 1-qubit circuit with Qiskit U3 (u gate) and return its Operator matrix."""
    qc = QuantumCircuit(1)
    qc.u(theta, phi, lam, 0)
    return Operator(qc).data

# Global-Phase-Invariant Unitary Distance
Two unitaries U and V represent the same physical quantum operation when:

    U = e^{iφ} V

because global phase e^{iφ} has no observable effect.

We use the distance metric:

    d(U,V) = 1 – |Tr(U V†)| / dim

Properties:
- d(U,V) = 0  ⟺  U = e^{iφ} V
- It is fast to compute
- It is mathematically robust
- It works even if two matrices only differ by floating-point roundoff

This function allows us to numerically check when

    U(θ, Φ, λ) ≈ H.

In [ ]:
def global_phase_invariant_distance(U: np.ndarray, V: np.ndarray) -> float:
    """
    Distance between unitaries up to global phase:
      d(U,V) = 1 - |Tr(U V†)| / dim
    This is 0 when U = e^{i phi} V.
    """
    dim = U.shape[0]
    tr = np.trace(U @ V.conj().T)
    return 1.0 - abs(tr) / dim

# Grid + Refinement Search for Best Params
Although we know no simple analytic solution for

    U(θ, Φ, λ) = H
this function demonstrates how one can *numerically discover* the values.

Why demonstrate numeric search?
- Useful in practice when analytic formulas are not obvious
- Mirrors calibration routines used in real quantum devices
- Shows how to match target gates through optimization methods

Process:
  1. Coarse grid search over [0, 2π) for θ, Φ, λ
     Evaluate d(U(θ, Φ, λ), H) for each combination.
     Keep the params with minimal distance.
  2. Refinement
     Take a small window (e.g., ±0.2 rad) around the best coarse values.
     Evaluate many param values in that 3D window for high precision.
  3. Normalize params back into [0, 2π).

In [ ]:
def find_params_gridrefine(num_grid=50, refine_radius=0.2, refine_steps=20):
    """
    Coarse-to-fine search for theta, phi, lambda in [0, 2*pi).
    Returns best params (in radians) and distance found.
    Note: num_grid^3 evaluations in coarse; refine_steps^3 in refinement.
    """
    thetas = np.linspace(0.0, 2 * math.pi, num_grid, endpoint=False)
    phis = np.linspace(0.0, 2 * math.pi, num_grid, endpoint=False)
    lams = np.linspace(0.0, 2 * math.pi, num_grid, endpoint=False)
    best_val = 1.0
    best_theta, best_phi, best_lam = None, None, None
    for th in thetas:
        for ph in phis:
            for la in lams:
                U = u_matrix_numpy(th, ph, la)
                val = global_phase_invariant_distance(U, H)
                if val < best_val:
                    best_val = val
                    best_theta, best_phi, best_lam = th, ph, la
    # Refine around best params
    center_t, center_p, center_l = best_theta, best_phi, best_lam
    low_t = center_t - refine_radius
    high_t = center_t + refine_radius
    low_p = center_p - refine_radius
    high_p = center_p + refine_radius
    low_l = center_l - refine_radius
    high_l = center_l + refine_radius
    thetas_ref = np.linspace(low_t, high_t, refine_steps)
    phis_ref = np.linspace(low_p, high_p, refine_steps)
    lams_ref = np.linspace(low_l, high_l, refine_steps)
    for th in thetas_ref:
        thm = ((th % (2 * math.pi)) + 2 * math.pi) % (2 * math.pi)
        for ph in phis_ref:
            phm = ((ph % (2 * math.pi)) + 2 * math.pi) % (2 * math.pi)
            for la in lams_ref:
                lam = ((la % (2 * math.pi)) + 2 * math.pi) % (2 * math.pi)
                U = u_matrix_numpy(thm, phm, lam)
                val = global_phase_invariant_distance(U, H)
                if val < best_val:
                    best_val = val
                    best_theta, best_phi, best_lam = thm, phm, lam
    return best_theta, best_phi, best_lam, best_val

# Validation Function: Action-Level + Unitary-Level Checks
This function verifies that U(θ, Φ, λ) matches H up to global phase in two ways:
1. **Unitary-Level Check**

   Using global-phase-invariant distance, compare:
      - U_numpy(θ, Φ, λ) vs H
      - U_qiskit(θ, Φ, λ) vs H
      
   to ensure consistency between analytic and Qiskit implementations.
2. **State-Action Check**
   Compare the action on |0⟩

        U(θ, Φ, λ)|0⟩
    vs

        e^{iφ} H|0⟩
    
   If they differ only by global phase, the gate is valid.
The state-action check extracts the relative phase between two vectors
and computes the norm of the difference.

Return:
- distances
- global phase
- boolean validity flag

In [ ]:
def validate_params(theta: float, phi: float, lam: float, tol=1e-9):
    """
    Validate that U(theta, phi, lambda) equals H up to global phase.
    Returns dict with distances and Qiskit-verified matrix equality.
    """
    U_np = u_matrix_numpy(theta, phi, lam)
    U_q = u_matrix_qiskit(theta, phi, lam)
    d_np = global_phase_invariant_distance(U_np, H)
    d_q = global_phase_invariant_distance(U_q, H)
    # Check action on basis states (up to phase)
    e0 = np.array([1.0, 0.0], dtype=complex)
    out_u_on_e0 = U_np @ e0
    out_h_on_e0 = H @ e0
    # Compute relative phase
    if np.linalg.norm(out_h_on_e0) > 0:
        s = (out_u_on_e0[0] / out_h_on_e0[0]) if out_h_on_e0[0] != 0 else (out_u_on_e0[1] / out_h_on_e0[1])
    else:
        s = 0
    action_error = np.linalg.norm(out_u_on_e0 - s * out_h_on_e0)
    return {
        "theta": theta,
        "phi": phi,
        "lambda": lam,
        "distance_unitary_np": float(d_np),
        "distance_unitary_qiskit": float(d_q),
        "action_error_on_|0>": float(action_error),
        "global_phase_scalar_on_|0>": complex(s),
        "is_valid": (d_np < tol and action_error < 1e-8)
    }

# Objective Function for Local Optimization
This function computes the global-phase-invariant distance for use as an objective in scipy's minimize. It takes a parameter vector [theta, phi, lambda] and returns
the scalar distance d(U(params), H).

Purpose:
- Enables gradient-free local optimization (e.g., Nelder-Mead) to polish
the grid search result.
- Fast to evaluate, as it reuses the distance metric.

In [ ]:
def distance_up_to_global_phase(params):
    """Scalar objective: global-phase-invariant distance between U(params) and H."""
    th, ph, la = params
    U = u_matrix_numpy(th, ph, la)
    # same distance you used
    dim = U.shape[0]
    tr = np.trace(U @ H.conj().T)
    return 1.0 - abs(tr) / dim

# Local Optimization Polish
This function refines the grid search result using scipy's Nelder-Mead optimizer.

Steps:
- Start from the coarse/refined params.
- Minimize the distance objective with tight tolerances.
- Wrap results into [0, 2π).

Purpose:
- Achieves machine-precision accuracy (distance ~1e-15).
- Complements grid search for cases where the minimum is narrow.
- Useful in gate-synthesis and calibration for high-fidelity matching.

In [ ]:
def polish_with_local_optimizer(theta0, phi0, lam0):
    """Run Nelder-Mead to refine the coarse solution."""
    x0 = np.array([theta0, phi0, lam0])
    res = minimize(distance_up_to_global_phase, x0, method='Nelder-Mead',
                   options={'xatol': 1e-12, 'fatol': 1e-12, 'maxiter': 5000, 'disp': False})
    th, ph, la = res.x
    # wrap into [0, 2pi)
    th = (th + 2*math.pi) % (2*math.pi)
    ph = (ph + 2*math.pi) % (2*math.pi)
    la = (la + 2*math.pi) % (2*math.pi)
    return th, ph, la, float(res.fun)

# Main Routine: Numeric Search → Polish → Validation

This block controls the execution when the script is run directly.

Steps:
1. Print problem label:

        U(θ, Φ, λ) = H  //up to global phase
2. Execute the numeric grid + refine search.
   This finds params very close to the true minimum.
3. Polish with local optimizer for machine precision.
4. Validate the polished params using:
      - unitary comparison
      - action-level comparison
5. Print the final conclusion.

This mirrors a realistic workflow:
- discover parameters numerically
- refine with optimization
- validate with exact math
- validate with Qiskit’s implementation

In [ ]:
if __name__ == "__main__":
    # ... your printing of Problem label ...
    label = LatexNodes2Text().latex_to_text(r"U(\theta,\Phi,\lambda) = H")
    print(f"Problem: {label}")
    theta_found, phi_found, lam_found, val = find_params_gridrefine()
    print(f"Coarse/refined numeric best (before polish): theta = {theta_found:.12f}, phi = {phi_found:.12f}, lambda = {lam_found:.12f}, dist = {val:.3e}")

    # NOW polish
    theta_p, phi_p, lam_p, val_p = polish_with_local_optimizer(theta_found, phi_found, lam_found)
    print(f"Polished (Nelder-Mead): theta = {theta_p:.12f}, phi = {phi_p:.12f}, lambda = {lam_p:.12f}, dist = {val_p:.3e}")

    # Validate the polished result
    result = validate_params(theta_p, phi_p, lam_p)
    print(f"Validation: distance_unitary_np = {result['distance_unitary_np']:.3e}, distance_unitary_qiskit = {result['distance_unitary_qiskit']:.3e}")
    print(f"Action on |0>: error = {result['action_error_on_|0>']:.3e}, global-phase ≈ {result['global_phase_scalar_on_|0>']:.12f}")
    print(f"Conclusion: valid = {result['is_valid']}")

Problem: U(θ,Φ,λ) = H
Coarse/refined numeric best (before polish): theta = 1.560596052670, phi = 6.272658991390, lambda = 3.152118969379, dist = 4.056e-05
Polished (Nelder-Mead): theta = 1.570796327409, phi = 0.000000009133, lambda = 3.141592651138, dist = 0.000e+00
Validation: distance_unitary_np = 1.110e-16, distance_unitary_qiskit = 1.110e-16
Action on |0>: error = 6.473e-09, global-phase ≈ 0.999999999693+0.000000000000j
Conclusion: valid = True


#Results
Running it will execute the full
optimization pipeline: grid search, refinement, polish, and validation.

Expected output includes params near theta=π, phi=0, lambda=π (or equivalents),
with distances ~1e-15 and valid=True.